In [14]:
import os
import cv2
import face_recognition
import pickle

DATASET_PATH = "dataset"
ENCODINGS_PATH = "encoded_faces.pkl"

def generate_encodings():
    known_encodings = []
    known_names = []

    if not os.path.exists(DATASET_PATH):
        print(f"❌ ERROR: '{DATASET_PATH}' folder not found!")
        return

    print("🚀 NEURAL-LINK: Scanning 'dataset' for new identities...")

    for person_name in os.listdir(DATASET_PATH):
        if person_name.startswith('.'):
            continue
            
        person_dir = os.path.join(DATASET_PATH, person_name)
        if not os.path.isdir(person_dir):
            continue

        print(f"🔍 Processing: {person_name}...")
        
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            
            image = cv2.imread(img_path)
            if image is None: 
                continue 
            
            # FIX: Moved to a new line
            rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # TRY 1: Standard HOG Scan
            face_locations = face_recognition.face_locations(rgb_image, model="hog")
            
            # TRY 2: Extreme Scan
            if not face_locations:
                print(f"    🔄 Standard failed. Attempting Extreme Scan on {img_name}...")
                face_locations = face_recognition.face_locations(rgb_image, number_of_times_to_upsample=3)

            if face_locations:
                encoding = face_recognition.face_encodings(rgb_image, face_locations)[0]
                known_encodings.append(encoding)
                known_names.append(person_name)
                print(f"    ✅ Success: {person_name} added.")
            else:
                print(f"    ❌ FAILED: Face invisible in {img_path}.")

    if known_encodings:
        with open(ENCODINGS_PATH, "wb") as f:
            pickle.dump({"encodings": known_encodings, "names": known_names}, f)
        print(f"\n✨ SUCCESS: Brain updated with {len(set(known_names))} identities.")
    else:
        print("\n🚫 CRITICAL: No faces found.")

generate_encodings()

🚀 NEURAL-LINK: Scanning 'dataset' for new identities...
🔍 Processing: Abhinav Singh...
    ✅ Success: Abhinav Singh added.
    ✅ Success: Abhinav Singh added.
🔍 Processing: Jatin Mehra...
    ✅ Success: Jatin Mehra added.
🔍 Processing: suneet kumar...
    ✅ Success: suneet kumar added.

✨ SUCCESS: Brain updated with 3 identities.


In [15]:
import pickle
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from datetime import datetime


def load_mask_detector(model_path):
    if os.path.exists(model_path):
        try:

            return load_model(model_path, compile=False)
        except Exception as e:
            print(f"❌ ERROR loading .h5: {e}")
            return None
    return None

def load_liveness_detector():
    print("📡 Liveness Module: Initialized (Active Blink Tracking Mode)")
    return True

ENCODINGS_PATH = "encoded_faces.pkl"
global known_encodings, known_names

if os.path.exists(ENCODINGS_PATH):
    with open(ENCODINGS_PATH, "rb") as f:
        data = pickle.load(f)
    known_encodings = data.get("encodings", [])
    known_names = data.get("names", [])
    print(f"🧠 NEURAL-LINK: Loaded {len(known_encodings)} face signatures.")
else:
    known_encodings = []
    known_names = []  
    print("⚠️ WARNING: No 'encoded_faces.pkl' found!")


MASK_MODEL_PATH = "models/mask_detector.h5" if os.path.exists("models/mask_detector.h5") else "mask_detector.h5"

mask_model = load_mask_detector(MASK_MODEL_PATH) 

if mask_model:
    print(f"✅ SUCCESS: Mask detection model loaded from {MASK_MODEL_PATH}")
else:
    print("❌ ERROR: Mask model (.h5) not found. System will fallback to landmark-based detection.")

liveness_loaded = load_liveness_detector() 


global checked_in_today, name_history, last_ear, blink_confirmed

checked_in_today = {} 
name_history = []     
last_ear = 0          
blink_confirmed = False 

🧠 NEURAL-LINK: Loaded 4 face signatures.
✅ SUCCESS: Mask detection model loaded from models/mask_detector.h5
📡 Liveness Module: Initialized (Active Blink Tracking Mode)


In [16]:
import threading
import time
from collections import Counter
import cv2
import numpy as np
import face_recognition
from scipy.spatial import distance as dist
import os
import pandas as pd
import tensorflow as tf


LOG_FILE = "attendance/attendance_log.csv"
RECOGNITION_TOLERANCE = 0.45 
MASKED_TOLERANCE = 0.60      
ATTENDANCE_COOLDOWN = 60  
scan_line_y = 0
scan_speed = 8 


frame_count = 0
is_obstructed = False 
blink_confirmed = False
last_ear = 0
last_face_time = 0
persistent_face_loc = None


PERSISTENCE_DURATION = 0.3  


def log_attendance(name):
    global checked_in_today, ATTENDANCE_COOLDOWN
    folder = "attendance"
    if not os.path.exists(folder): os.makedirs(folder)
    now = time.time()
    dt_string = time.strftime('%Y-%m-%d %H:%M:%S')
    
    if name in checked_in_today and (now - checked_in_today[name]) < ATTENDANCE_COOLDOWN:
        return "ALREADY_LOGGED"

    new_row = pd.DataFrame([[name, dt_string, "Present"]], columns=['Name', 'Timestamp', 'Status'])
    file_exists = os.path.isfile(LOG_FILE)
    new_row.to_csv(LOG_FILE, mode='a', index=False, header=not file_exists)
    checked_in_today[name] = now
    return "SUCCESS"

def detect_strict_blink(landmarks):
    global last_ear, blink_confirmed
    def eye_aspect_ratio(eye):
        A = dist.euclidean(eye[1], eye[5])
        B = dist.euclidean(eye[2], eye[4])
        C = dist.euclidean(eye[0], eye[3])
        return (A + B) / (2.0 * C)
    
    leftEye = landmarks.get('left_eye')
    rightEye = landmarks.get('right_eye')
    
    if leftEye and rightEye:
        current_ear = (eye_aspect_ratio(leftEye) + eye_aspect_ratio(rightEye)) / 2.0
        if last_ear > 0.18 and current_ear < 0.15:
            blink_confirmed = True
        last_ear = current_ear
    return blink_confirmed


def draw_ai_hud(frame, face_loc, display_name, status_msg, state_color):
    global scan_line_y
    top, right, bottom, left = face_loc
    yellow = (0, 255, 255) 
    magenta = (255, 0, 255)

    scan_line_y += scan_speed
    if scan_line_y < top or scan_line_y > bottom:
        scan_line_y = top
    cv2.line(frame, (left + 5, scan_line_y), (right - 5, scan_line_y), yellow, 1)

    d, t = 35, 2
    cv2.line(frame, (left, top), (left + d, top), state_color, t)
    cv2.line(frame, (left, top), (left, top + d), state_color, t)
    cv2.line(frame, (right, top), (right - d, top), state_color, t)
    cv2.line(frame, (right, top), (right, top + d), state_color, t)
    cv2.line(frame, (left, bottom), (left + d, bottom), state_color, t)
    cv2.line(frame, (left, bottom), (left, bottom - d), state_color, t)
    cv2.line(frame, (right, bottom), (right - d, bottom), state_color, t)
    cv2.line(frame, (right, bottom), (right, bottom - d), state_color, t)

    cv2.rectangle(frame, (left, top - 45), (right, top - 5), state_color, -1)
    cv2.putText(frame, display_name.upper(), (left + 12, top - 18), 
                cv2.FONT_HERSHEY_DUPLEX, 0.6, (0, 0, 0), 1)

    cv2.putText(frame, f">> {status_msg}", (left, bottom + 35), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, magenta, 1)


class WebcamStream:
    def __init__(self, src=0):
        self.stream = cv2.VideoCapture(src)
        self.grabbed, self.frame = self.stream.read()
        self.stopped = False
    def start(self):
        threading.Thread(target=self.update, args=(), daemon=True).start()
        return self
    def update(self):
        while not self.stopped:
            self.grabbed, self.frame = self.stream.read()
    def read(self): return self.frame
    def stop(self):
        self.stopped = True
        self.stream.release()


vs = WebcamStream(src=0).start()
time.sleep(2.0)

current_display_name = "INITIALIZING..."
current_status = "READY"
current_color = (255, 0, 0) 

try:
    while True:
        frame = vs.read()
        if frame is None: break
        frame_count += 1
        
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
        rgb_small = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
        
        face_locs = face_recognition.face_locations(rgb_small, model="hog")
        
        if face_locs:
            last_face_time = time.time()
            persistent_face_loc = [c * 4 for c in face_locs[0]]
        
        # Check if the persistence window has closed
        if not face_locs and (time.time() - last_face_time > PERSISTENCE_DURATION):
            blink_confirmed = False
            last_ear = 0
            current_display_name = "SCANNING..."
            current_color = (255, 0, 0)
            cv2.imshow("NEURAL-LINK AI CORE", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
            continue

        if face_locs and persistent_face_loc:
            face_landmarks = face_recognition.face_landmarks(small_frame, face_locs)
            if face_landmarks:
                is_live = detect_strict_blink(face_landmarks[0])
            
            if frame_count % 5 == 0:
                if 'known_encodings' in globals():
                    face_encs = face_recognition.face_encodings(small_frame, face_locs)
                    if face_encs:
                        matches = face_recognition.compare_faces(known_encodings, face_encs[0], tolerance=RECOGNITION_TOLERANCE)
                        
                        if True in matches:
                            match_name = known_names[matches.index(True)]
                            log_status = log_attendance(match_name)

                            if log_status == "ALREADY_LOGGED":
                                current_display_name = match_name
                                current_status = "THANK YOU - PREVIOUSLY LOGGED"
                                current_color = (0, 255, 0) 
                            else:
                                if is_live:
                                    current_display_name = match_name
                                    current_status = "IDENTITY VERIFIED"
                                    current_color = (0, 255, 0)
                                else:
                                    current_display_name = "USER DETECTED"
                                    current_status = "LIVENESS CHECK: BLINK NOW"
                                    current_color = (0, 255, 255) 
                        else:
                            current_display_name = "UNKNOWN"
                            current_status = "UNAUTHORIZED ACCESS"
                            current_color = (0, 0, 255) 

        if face_locs or (time.time() - last_face_time <= PERSISTENCE_DURATION):
            if persistent_face_loc:
                draw_ai_hud(frame, persistent_face_loc, current_display_name, current_status, current_color)

        cv2.imshow("NEURAL-LINK AI CORE", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

finally:
    vs.stop()
    cv2.destroyAllWindows()